# Brazilian Tourism Data Pipeline (1989-2024)

**Source:** [Ministério do Turismo - Dados Abertos](https://dados.gov.br/dados/conjuntos-dados/estimativas-de-chegadas-de-turistas-internacionais-ao-brasil)

## Project Overview
This project demonstrates the end-to-end process of handling real-world historical data. 
It focuses on building a robust ETL (Extract, Transform, Load) pipeline to consolidate 
over 30 years of Brazilian tourism records, addressing challenges like schema drift 
and data inconsistency.

---

## Table of Contents
0. [Setup](#setup)
1. [Schema Discovery](#schema-discovery)
2. [Mapping Strategy](#normalization)
3. [Data Pipeline](#pipeline)<br>
&nbsp;3.1. [Transformation Logic](#logic)<br>
&nbsp;3.2. [Unit Test & Quality Audit](#quality-audit)<br>
&nbsp;3.3. [Data Aggregation](#aggregation)
4. [Quality Check](#quality-check)<br>
&nbsp;4.1. [Checking NaN Values](#nancheck)<br>
&nbsp;4.2. [Data Audit](#data-audit)<br>
&nbsp;4.3. [Data Quality Manifesto & Imputation Transparency](#data-manifesto)<br>
5. [Database Export](#database)<br>
6. [Post-Export Sanity Check](#final-check)

## 0. Environment Setup & Configuration <a id="setup"></a>
Initializes necessary libraries and defines global settings for data handling and visualization.

In [1]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path

pd.set_option('display.max_columns', None)

## 1. Schema Discovery <a id="schema-discovery"></a>
The first step of our pipeline is to "scan" the raw files. Since this dataset spans 
decades, column names often change. We identify these variations to build a 
mapping strategy without loading the entire dataset into memory.


In [2]:
def get_data_inventory(directory_path: str, separator: str = ';'):
    """
    Scans all CSV files to check if their structure match.
    Returns a list of dictionaries with the different structure among all files.
    """
    path = Path(directory_path)
    files = sorted(list(path.glob("chegadas_*.csv")))
    
    inventory = []
    
    for f in files:
        # Read only the header for efficiency
        header_df = pd.read_csv(f, sep=separator, nrows=0, encoding='latin1')
        cols = header_df.columns.tolist()
        
        inventory.append({
            "filename": f.name,
            "col_count": len(cols),
            "columns": cols
        })
        
    return inventory

In [3]:
# Path to raw data
raw_folder = "../data/raw/"

# Execute inventory scan
files_inventory = get_data_inventory(raw_folder)
inventory_df = pd.DataFrame(files_inventory)

# Filter to show only the unique schemas found
pd.set_option('display.max_colwidth', None)
inventory_df['cols_str'] = inventory_df['columns'].astype(str)

unique_schemas = (
    inventory_df
    .drop_duplicates(subset=['col_count', 'cols_str'])
    .drop(columns=['cols_str'])
)

print(f"{len(unique_schemas)} different types of file structures have been found.\n")
unique_schemas

3 different types of file structures have been found.



,filename,col_count,columns
0,chegadas_1989.csv,12,"[Continente, Ordem continente, País, Ordem país, UF, Ordem UF, Via de acesso, Ordem via de acesso, ano, Mês, Ordem mês, Chegadas]"
11,chegadas_2000.csv,12,"[Continente, Ordem continente, País, Ordem país, UF, Ordem UF, Via de acesso, Ordem via de acesso, Ano, Mês, Ordem mês, Chegadas]"
27,chegadas_2016.csv,12,"[Continente, cod continente, País, cod pais, UF, cod uf, Via, cod via, ano, Mês, cod mes, Chegadas]"


### Analysis of Findings
All files consistently contain **12 columns**, which simplifies the consolidation. However, the inventory reveals three distinct patterns:

* **Pattern 1 & 2:** The primary difference is simple capitalization (e.g., `Ano` vs. `ano`).
* **Pattern 3:** Column names are significantly shortened, and the capitalization remains inconsistent.

These findings confirm that while the "shape" of the data is stable, a **mapping layer** is essential to ensure that columns from different years align perfectly during the merge.

## 2. Mapping Strategy <a id="normalization"></a>

The next step is to standardize our dataset. We will define our "Source of Truth" by choosing official column names and ensuring every year follows the same schema. 

Additionally, we will eliminate redundant data (like auxiliary codes) identified during the initial analysis of the 1989 file, keeping only what is essential for analysis.

In [4]:
def standardize():
    """
    Returns the mapping dictionary and the list of columns to keep.
    This unifies naming patterns and removes redundant data.
    """
    column_mapping = {
        'Continente': 'continent',
        'País': 'country',
        'UF': 'state',
        'Via de acesso': 'arrival_method', 'Via': 'arrival_method',
        'ano': 'year', 'Ano': 'year',
        'Mês': 'month',
        'Chegadas': 'arrivals'
    }
    target_columns = ['continent', 'country', 'state', 'arrival_method', 'year', 'month', 'arrivals']
    
    return column_mapping, target_columns

In [5]:
# Initialize mapping rules
column_map, target_cols = standardize()

# --- Proof of Concept (Sanity Check) ---
# Testing if the mapping correctly unifies the unique structures from Step 1

def apply_test_mapping(cols_list):
    # Rename and filter in one step for the test
    return [column_map.get(c) for c in cols_list if column_map.get(c) in target_cols]

# Apply the test to unique_schemas table
unique_schemas['standardized_columns'] = unique_schemas['columns'].apply(apply_test_mapping)

# Display result
print("Verification: If all rows in 'standardized_columns' are identical, the mapping is successful.")
unique_schemas[['filename', 'standardized_columns']]

Verification: If all rows in 'standardized_columns' are identical, the mapping is successful.


,filename,standardized_columns
0,chegadas_1989.csv,"[continent, country, state, arrival_method, year, month, arrivals]"
11,chegadas_2000.csv,"[continent, country, state, arrival_method, year, month, arrivals]"
27,chegadas_2016.csv,"[continent, country, state, arrival_method, year, month, arrivals]"


### Strategy Validation
The Proof of Concept was **successful**. Even with different source schemas, the mapping function consistently produced the same set of 7 target columns. 

**Next steps:**
* Transition from header-only analysis to full file processing.
* Loop through the raw directory to consolidate all years into a single master dataset.

## 3. Data Pipeline <a id="pipeline"></a>
Now we apply our strategy to the actual data. This process is divided into the transformation of individual files and their final consolidation.

### 3.1. Transformation Logic <a id="logic"></a>

In [6]:
def transform_data(file_path, column_map, target_cols):
    """
    Reads a single file, applies renaming, filters columns, 
    and handles basic data cleaning and string standardization.
    """
    # 1. Extraction (Reading)
    df = pd.read_csv(file_path, sep=';', encoding='latin1')
    
    # 2. Transformation: Column Mapping
    df = df.rename(columns=column_map)
    
    # 3. Transformation: Filtering only target columns
    df = df[df.columns.intersection(target_cols)].copy()
    
    # 4. Data Cleansing: String Standardization
    categorical_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in categorical_cols:
        df[col] = df[col].astype(str).str.strip().str.lower()
    
    # 5. Data Cleansing: Numeric Conversion
    if 'arrivals' in df.columns:
        df['arrivals'] = pd.to_numeric(df['arrivals'], errors='coerce')
    
    # 6. Add metadata for traceability
    df['source_file'] = Path(file_path).name
    
    return df

### 3.2. Unit Test & Quality Audit <a id="quality-audit"></a>

Unit Test: 1989 Data
Before running the pipeline through all 35+ files, we perform a **Unit Test** on the 1989 dataset. This year is known for having inconsistent formatting and missing values.

Our goal here is to:
* Identify how many rows contain non-numeric "garbage" in the `arrivals` column.
* Verify if the column mapping correctly captured the historical headers.

In [7]:
# Select file for testing
test_file = "../data/raw/chegadas_1989.csv"

# Execute transformation (Injecting our rules)
df_1989_clean = transform_data(test_file, column_map, target_cols)

# Quality Health Report (Anomaly Detection)
nan_count = df_1989_clean['arrivals'].isna().sum()
total_rows = len(df_1989_clean)

print(f"--- Quality Report: {test_file} ---")
print(f"Total rows processed: {total_rows}")
print(f"Anomalies found (NaN in 'arrivals'): {nan_count}")
print(f"Error Rate: {(nan_count/total_rows)*100:.2f}%")
print("-" * 53)

# Preview transformed data
display(df_1989_clean.head())

--- Quality Report: ../data/raw/chegadas_1989.csv ---
Total rows processed: 17052
Anomalies found (NaN in 'arrivals'): 588
Error Rate: 3.45%
-----------------------------------------------------


,continent,country,state,arrival_method,year,month,arrivals,source_file
0,áfrica,áfrica do sul,amazonas,aérea,1989,janeiro,9.0,chegadas_1989.csv
1,áfrica,angola,amazonas,aérea,1989,janeiro,0.0,chegadas_1989.csv
2,áfrica,nigéria,amazonas,aérea,1989,janeiro,0.0,chegadas_1989.csv
3,áfrica,outros países,amazonas,aérea,1989,janeiro,0.0,chegadas_1989.csv
4,américa central e caribe,costa rica,amazonas,aérea,1989,janeiro,6.0,chegadas_1989.csv


### Unit Test Findings & Data Quality Audit
The transformation function is designed to preserve data integrity: where arrivals cannot be parsed as numeric, values are kept as NaN to avoid injecting bias. This is confirmed by the Unit Test below.
The test on 1989 data reveals:

Exactly 588 rows failed numeric conversion (3.45% of the dataset) <br>
Root cause: historical records for 'Mato Grosso do Sul' (Fluvial access) contain non-numeric characters <br>
Pipeline resilience: the coerce strategy isolates these anomalies without crashing the pipeline, keeping dirty data visible for later decision-making

### 3.3. Data Aggregation <a id='aggregation'></a>

With the transformation logic validated by our Unit Test, we now proceed to aggregate the entire historical series (1989-2024).

In [8]:
# List all CSV files in the raw data folder
file_paths = glob.glob("../data/raw/*.csv")

# List to store each cleaned DataFrame
all_dfs = []

print(f"Starting aggregation of {len(file_paths)} files...")

# Loop through files and transform
for file in file_paths:
    try:
        temp_df = transform_data(file, column_map, target_cols)
        all_dfs.append(temp_df)
    except Exception as e:
        print(f"Error processing file {file}: {e}")

# Concatenate everything into one single DataFrame
df_tourism = pd.concat(all_dfs, ignore_index=True)

print("Aggregation complete!")
print(f"Final dataset shape: {df_tourism.shape}")

# Preview the consolidated data
df_tourism.sample(5)

Starting aggregation of 36 files...
Aggregation complete!
Final dataset shape: (953672, 8)


,continent,country,state,arrival_method,year,month,arrivals,source_file
910749,américa central e caribe,outros países,santa catarina,aéreo,2023,agosto,0.0,chegadas_2023.csv
199015,ásia,outros países,pernambuco,marítima,2000,junho,7.0,chegadas_2000.csv
354149,américa do sul,argentina,santa catarina,aérea,2008,março,7600.0,chegadas_2008.csv
936953,europa,itália,outras unidades da federação,marítima,2024,março,1.0,chegadas_2024.csv
237677,europa,alemanha,mato grosso do sul,terrestre,2002,março,19.0,chegadas_2002.csv


In [9]:
df_tourism.info()

<class 'pandas.DataFrame'>
RangeIndex: 953672 entries, 0 to 953671
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   continent       953672 non-null  str    
 1   country         953672 non-null  str    
 2   state           953672 non-null  str    
 3   arrival_method  953672 non-null  str    
 4   year            953672 non-null  int64  
 5   month           953672 non-null  str    
 6   arrivals        943820 non-null  float64
 7   source_file     953672 non-null  str    
dtypes: float64(1), int64(1), str(6)
memory usage: 116.6 MB


The ETL pipeline has successfully standardized and consolidated the historical series from 1989 to 2024 into a single canonical structure.

**Final Dataset Overview:**
- **Dimensions:** 953,672 rows and 8 columns.
- **Structural Integrity:** 100% of the dimension columns (`continent`, `country`, `state`, `arrival_method`, `year`, `month`) are populated and standardized.
- **Data Quality Audit:** The `arrivals` column contains **~9,852 null values** (representing ~1.03% of the total volume). These gaps are isolated to specific legacy records and have been preserved as `NaN` to maintain statistical honesty for the upcoming analysis.

**Conclusion:**
The dataset is now structurally sound and optimized for the next stages of Data Quality refinement and Database integration.

## 4. Data Quality & Cleaning <a id="quality-check"></a>
To make sure data is consistent and identify possible mistakes, we will analyze the missing values on arrivals, which is our main column for analysis, but also we will make sure that all values on the other columns are consistent and aligned.
Why? Even if it's only ~1.03% of total values, it can still be a huge amount of data for determined year, so it's important to check how much impact that value really has before making decisions.

### 4.1. Checking NaN Values <a id="nancheck"></a>

Before converting `NaN` values to zeros, we perform a root cause analysis.
We verify if the missing data pattern observed in 1989 (Fluvial access in specific states) persists throughout the years.

In [10]:
def audit_missing_values(df):
    """
    Performs a forensic audit on missing values (NaN) in the tourism dataset.
    Identifies problematic years, percentages, and likely causes by grouping
    dimensions (State, Arrival Method, etc.)
    """
    
    # General identification of years with nulls
    null_data = df[df['arrivals'].isna()]
    years_with_nulls = null_data['year'].unique()
    
    if len(years_with_nulls) == 0:
        print("No missing values found in the 'arrivals' column.")
        return None

    report_list = []

    for year in sorted(years_with_nulls):
        yearly_df = df[df['year'] == year]
        total_rows = len(yearly_df)
        nan_rows = yearly_df['arrivals'].isna().sum()
        nan_pct = (nan_rows / total_rows) * 100
        
        # Investigate specific causes:
        # Which states are failing?
        missing_states = yearly_df[yearly_df['arrivals'].isna()]['state'].unique()
        
        # Deep Dive: Is it a specific combination?
        # We group to see if a state lost 100% of its data or just one category
        cause_summary = []
        for state in missing_states:
            state_data = yearly_df[yearly_df['state'] == state]
            state_nan_cnt = state_data['arrivals'].isna().sum()
            state_total = len(state_data)
            
            if state_nan_cnt == state_total:
                cause_summary.append(f"State '{state.upper()}' registered NO data (100% missing)")
            else:
                methods = state_data[state_data['arrivals'].isna()]['arrival_method'].unique()
                cause_summary.append(f"State '{state.upper()}' missing methods: {list(methods)}")

        report_list.append({
            'Year': year,
            'Missing %': round(nan_pct, 2),
            'Total NaN': nan_rows,
            'Affected States': list(missing_states),
            'Likely Causes': " | ".join(cause_summary)
        })

    # Convert to DataFrame for a polished display
    report_df = pd.DataFrame(report_list)
    return report_df

In [11]:
# Running it
quality_report = audit_missing_values(df_tourism)
display(quality_report)

,Year,Missing %,Total NaN,Affected States,Likely Causes
0,1989,3.45,588,[mato grosso do sul],State 'MATO GROSSO DO SUL' missing methods: ['fluvial']
1,1996,9.68,1764,[santa catarina],State 'SANTA CATARINA' registered NO data (100% missing)
2,1999,3.23,588,[distrito federal],State 'DISTRITO FEDERAL' registered NO data (100% missing)
3,2004,9.26,1920,"[amazonas, bahia, ceará, pará, paraná, pernambuco, rio grande do norte, rio grande do sul, rio de janeiro, santa catarina, são paulo, outras unidades da federação, mato grosso do sul]","State 'AMAZONAS' missing methods: ['aérea', 'terrestre'] | State 'BAHIA' missing methods: ['aérea', 'marítima'] | State 'CEARÁ' missing methods: ['aérea', 'marítima'] | State 'PARÁ' missing methods: ['aérea', 'fluvial'] | State 'PARANÁ' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'PERNAMBUCO' missing methods: ['aérea', 'marítima'] | State 'RIO GRANDE DO NORTE' missing methods: ['aérea', 'marítima'] | State 'RIO GRANDE DO SUL' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'RIO DE JANEIRO' missing methods: ['aérea', 'marítima'] | State 'SANTA CATARINA' missing methods: ['aérea', 'marítima', 'terrestre'] | State 'SÃO PAULO' missing methods: ['aérea', 'marítima'] | State 'OUTRAS UNIDADES DA FEDERAÇÃO' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'MATO GROSSO DO SUL' missing methods: ['terrestre']"
4,2007,10.71,2304,"[amazonas, bahia, ceará, pará, paraná, pernambuco, rio grande do norte, rio grande do sul, rio de janeiro, santa catarina, são paulo, outras unidades da federação, mato grosso do sul]","State 'AMAZONAS' missing methods: ['aérea', 'terrestre'] | State 'BAHIA' missing methods: ['aérea', 'marítima'] | State 'CEARÁ' missing methods: ['aérea', 'marítima'] | State 'PARÁ' missing methods: ['aérea', 'fluvial'] | State 'PARANÁ' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'PERNAMBUCO' missing methods: ['aérea', 'marítima'] | State 'RIO GRANDE DO NORTE' missing methods: ['aérea', 'marítima'] | State 'RIO GRANDE DO SUL' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'RIO DE JANEIRO' missing methods: ['aérea', 'marítima'] | State 'SANTA CATARINA' missing methods: ['aérea', 'marítima', 'terrestre'] | State 'SÃO PAULO' missing methods: ['aérea', 'marítima'] | State 'OUTRAS UNIDADES DA FEDERAÇÃO' missing methods: ['aérea', 'marítima', 'terrestre', 'fluvial'] | State 'MATO GROSSO DO SUL' missing methods: ['terrestre']"
5,2012,8.82,2016,"[paraná, rio grande do sul]","State 'PARANÁ' missing methods: ['marítima', 'fluvial'] | State 'RIO GRANDE DO SUL' missing methods: ['marítima']"
6,2014,2.44,672,[mato grosso do sul],State 'MATO GROSSO DO SUL' missing methods: ['fluvial']


### Analysis & Decision on Missing Values

After running the automated audit, three distinct patterns of missing data were identified:

1. **Localized Method Gaps (e.g., 1989, 2014):** Specific arrival methods (mostly `Fluvial`) were not recorded for `Mato Grosso do Sul`. Since other methods were documented, these values are likely zero or negligible. 
    * **Action:** Values were filled with `0`.

2. **State-Wide Blackouts (e.g., 1996 - SC, 1999 - DF):** Entire states failed to report arrivals for a full year. This represents a collection failure rather than a lack of tourists.
    * **Action:** Decided to fill these with `0` to maintain computational consistency, but a **disclaimer** will be added to final visualizations to prevent misinterpreting these gaps as actual drops in demand.

3. **Systemic Collection Shifts (2004 & 2007):** A high percentage of missing data (~10%) across multiple states suggests a migration issue in the original source or a change in the reporting system.
    * **Action:** Filled with `0` and flagged as statistical outliers for trend analysis.

#### Final Decision:
To ensure the integrity of mathematical operations, all `NaN` values were converted to `0`. A `data_quality_flag` column will be implemented to maintain data lineage, allowing for transparent reporting in the dashboard phase.

### Cleaning NaN Values

In [12]:
# 1. Creating a flag to preserve data lineage before filling
df_tourism['data_quality_flag'] = np.where(df_tourism['arrivals'].isna(), 'imputed_zero', 'original')

# 2. Filling NaNs with 0
df_tourism['arrivals'] = df_tourism['arrivals'].fillna(0).astype(int)

# 3. Final Verification
print(f"Current NaN count: {df_tourism['arrivals'].isna().sum()}")
print(df_tourism['data_quality_flag'].value_counts())

Current NaN count: 0
data_quality_flag
original        943820
imputed_zero      9852
Name: count, dtype: int64


### 4.2. Data Audit <a id="data-audit"></a>

Now NaN values were handled, the next thing to verify is data integrity, we will check if there are any easily identifiable year typos, also will check all other columns data to look for typos, writing variations, and other common mistakes, for this step a function will be created to allow checking all the values, but we will need to check it by eye to identify possible mistakes.

In [13]:
def data_audit(df):
    """
    Performs a comprehensive audit of the dataset:
    1. Numerical range check for 'year'
    2. Categorical consistency for all string columns
    3. Detection of leading/trailing whitespaces
    """
    
    print(f"STARTING DATA AUDIT")

    # Year Range Audit
    y_min, y_max = df['year'].min(), df['year'].max()
    print(f"\n[TIME RANGE] Data spans from {y_min} to {y_max}")

    # Categorical Audit
    cols_to_audit = ['continent', 'country', 'state', 'arrival_method', 'month']
    
    for col in cols_to_audit:
        unique_values = df[col].unique()
        # Check for whitespaces
        whitespaces = [val for val in unique_values if str(val) != str(val).strip()]
        
        print(f"\n[COLUMN: {col.upper()}]")
        print(f"   - Unique values found: {len(unique_values)}")
        
        if len(whitespaces) > 0:
            print(f"   - ALERT: Found {len(whitespaces)} values with hidden whitespaces!")

        # Showing all values, even if there is a huge amount of data, we need to check all of them to identify possible duplicates
        all_values = sorted([str(x) for x in unique_values])
        print(f"   - Values: {all_values}")

    print(f"Audit Complete!")

In [14]:
# Running the audit
data_audit(df_tourism)

STARTING DATA AUDIT

[TIME RANGE] Data spans from 1989 to 2024

[COLUMN: CONTINENT]
   - Unique values found: 8
   - Values: ['américa central e caribe', 'américa do norte', 'américa do sul', 'continente não especificado', 'europa', 'oceania', 'áfrica', 'ásia']

[COLUMN: COUNTRY]
   - Unique values found: 96
   - Values: ['alemanha', 'angola', 'argentina', 'arábia saudita', 'austrália', 'bangladesh', 'bolívia', 'bulgária', 'bélgica', 'cabo verde', 'canadá', 'chile', 'china', 'china, hong kong', 'cingapura', 'colômbia', 'costa rica', 'croácia', 'cuba', 'dinamarca', 'egito', 'el salvador', 'equador', 'eslováquia', 'eslovênia', 'espanha', 'estados unidos', 'estônia', 'filipinas', 'finlândia', 'frança', 'gana', 'grécia', 'guatemala', 'guiana', 'guiana francesa', 'haiti', 'holanda', 'honduras', 'hungria', 'indonésia', 'iraque', 'irlanda', 'irã', 'israel', 'itália', 'japão', 'letônia', 'lituânia', 'luxemburgo', 'líbano', 'malásia', 'marrocos', 'moçambique', 'méxico', 'nicarágua', 'nigéria', 

#### Data audit Analysis

Some categories like `Continente Não Especificado`, `Outros Países` and variations, `Outras Unidades Da Federação` will be kept in the data, but will be removed further depending on the type of query we will be doing as it can group multiple locations and have inflated values.

- Years: Everything seems right, data ranges from 1989 to 2024. 
- Continents: In continents nothing seems weird, we have all Continents except Antartida, and none of them seem to be repeated, American continent was divided into three, `America Central and Caribe`, `America Do Norte` and `America Do Sul`.
- Countries: In countries everything seems normal too, we can see `China`and `China, Honk Kong`, but decided to keep it as they can have different profiles of tourists, and there are many variations of `Outros Países` separated by continents which is good, there is one instance that isn't that works likely the one in 1989 analysis, that groups all different continents and is the one we will need to be careful with on further analysis.
- States: Brazil actually has 27 UF, the data shows 17 plus `Outras Unidades Da Federação`, we can for now assume it's due to all others having a very low number of tourists, but we will need to be careful in the future to see if our assumption really is right or not, in general it's not good to assume.
- Arrival Method: Here we can see the first mistake, the ones we were looking for, we found 6 different methods, while we knew about 4 so far, so there could be new ones, or there are typos or writing variants. Checking the values we can see `aérea` and `aéreo`, also `marítima` and `marítimo` to correct that we will be defining a single one to unite both arrivals methods.
- Months: 13 different months with unique values found, we know there are only 12 months. Checking values we can see variations of `marco` and `março` we will also be uniting both.

In [15]:
# Defining the semantic normalization dictionary
semantic_normalization = {
    'arrival_method': {
        'aérea': 'aéreo',
        'marítima': 'marítimo',
        'maritmo': 'marítimo' 
    },
    'month': {
        'marco': 'março'
    }
}

# Applying corrections across the dataframe
for col, mapping in semantic_normalization.items():
    df_tourism[col] = df_tourism[col].replace(mapping)

# Capitalization Standardization (Title Case)
cols_to_title = ['state', 'arrival_method', 'month', 'continent', 'country']
for col in cols_to_title:
    df_tourism[col] = df_tourism[col].str.title()

# Final Sanity Check
print("--- Final Validation ---")
print(f"Unique Methods: {df_tourism['arrival_method'].unique()}")
print(f"Unique Months (Count): {len(df_tourism['month'].unique())}")
if len(df_tourism['month'].unique()) == 12:
    print("Month consistency verified (12 unique months).")

--- Final Validation ---
Unique Methods: <ArrowStringArray>
['Aéreo', 'Terrestre', 'Fluvial', 'Marítimo']
Length: 4, dtype: str
Unique Months (Count): 12
Month consistency verified (12 unique months).


This validation allows us to check now the two problems we had were fixed, we have 4 unique `arrival_method`, and 12 unique `month` now.

In [16]:
# Final verification after all cleaning and normalization steps
display(df_tourism.sample(5))
data_audit(df_tourism)

,continent,country,state,arrival_method,year,month,arrivals,source_file,data_quality_flag
945838,Europa,Bulgária,Outras Unidades Da Federação,Marítimo,2024,Agosto,1,chegadas_2024.csv,original
815554,Europa,República Tcheca,Pará,Terrestre,2021,Novembro,0,chegadas_2021.csv,original
479965,Oceania,Nova Zelândia,Amazonas,Terrestre,2014,Maio,0,chegadas_2014.csv,original
597759,Ásia,República Da Coreia,Outras Unidades Da Federação,Marítimo,2016,Setembro,1,chegadas_2016.csv,original
608424,América Central E Caribe,Outros Países,Distrito Federal,Aéreo,2017,Abril,6,chegadas_2017.csv,original


STARTING DATA AUDIT

[TIME RANGE] Data spans from 1989 to 2024

[COLUMN: CONTINENT]
   - Unique values found: 8
   - Values: ['América Central E Caribe', 'América Do Norte', 'América Do Sul', 'Continente Não Especificado', 'Europa', 'Oceania', 'África', 'Ásia']

[COLUMN: COUNTRY]
   - Unique values found: 96
   - Values: ['Alemanha', 'Angola', 'Argentina', 'Arábia Saudita', 'Austrália', 'Bangladesh', 'Bolívia', 'Bulgária', 'Bélgica', 'Cabo Verde', 'Canadá', 'Chile', 'China', 'China, Hong Kong', 'Cingapura', 'Colômbia', 'Costa Rica', 'Croácia', 'Cuba', 'Dinamarca', 'Egito', 'El Salvador', 'Equador', 'Eslováquia', 'Eslovênia', 'Espanha', 'Estados Unidos', 'Estônia', 'Filipinas', 'Finlândia', 'França', 'Gana', 'Grécia', 'Guatemala', 'Guiana', 'Guiana Francesa', 'Haiti', 'Holanda', 'Honduras', 'Hungria', 'Indonésia', 'Iraque', 'Irlanda', 'Irã', 'Israel', 'Itália', 'Japão', 'Letônia', 'Lituânia', 'Luxemburgo', 'Líbano', 'Malásia', 'Marrocos', 'Moçambique', 'México', 'Nicarágua', 'Nigéria', 

### 4.3. Data Quality Manifesto & Imputation Transparency <a id="data-manifesto"></a>
As part of our data governance strategy, we track every record that was not present in the original source. Below is a summary of the data we imputed (filled with 0). 

**Heads-up for Analysts:** Years/States listed below had original missing values (likely due to reporting blackouts). Use the `data_quality_flag` to filter these out if high precision is required for local trend analysis.

In [17]:
# Filtering records where data was imputed
imputed_data_summary = df_tourism[df_tourism['data_quality_flag'] == 'imputed_zero']

if not imputed_data_summary.empty:
    print(f"TOTAL IMPUTED RECORDS: {len(imputed_data_summary)}")
    print("-" * 30)
    # Grouping to show where the gaps were
    report = imputed_data_summary.groupby(['year', 'state']).agg(
        imputed_records_count=('arrivals', 'count'),
        affected_methods=('arrival_method', 'nunique')
    ).reset_index()
    display(report)
else:
    print("No imputed data found. Dataset integrity: 100%")

TOTAL IMPUTED RECORDS: 9852
------------------------------


,year,state,imputed_records_count,affected_methods
0,1989,Mato Grosso Do Sul,588,1
1,1996,Santa Catarina,1764,3
2,1999,Distrito Federal,588,1
3,2004,Amazonas,120,2
4,2004,Bahia,120,2
5,2004,Ceará,120,2
6,2004,Mato Grosso Do Sul,60,1
7,2004,Outras Unidades Da Federação,240,4
8,2004,Paraná,240,4
9,2004,Pará,120,2


## 5. Data Export <a id="database"></a>

Once we finished transforming the data, we can load it different formats or structures depending on which case we want to use it. Here are some methods we will use.

1.  **Analytical Layer (Parquet):** Stored in columnar format with Hive-style partitioning by `Year`. This creates an optimized structure for Big Data engines to query specific years without scanning the full dataset.
2.  **Relational Layer (SQL):** Loaded into a SQLite database to simulate a Data Warehouse for transactional queries and BI tools.
3.  **Interoperability Layer (CSV):** Two formats — a compressed .csv.gz for sharing with non-technical stakeholders or legacy systems, and a plain .csv optimized for direct BI tool integration.

In [18]:
import sqlite3
import os
import shutil

# Define output paths
processed_path = "../data/processed"
parquet_path = f"{processed_path}/parquet_partitioned"
db_path = f"{processed_path}/tourism_dw.db"

# Create directory if it doesn't exist
os.makedirs(processed_path, exist_ok=True)

# 1. PARQUET EXPORT (Big Data / Analytics Optimized)
print("1. Exporting to Parquet ...")

# Cleaning previous exports to avoid duplication during multiple runs
if os.path.exists(parquet_path):
    shutil.rmtree(parquet_path)

# Writing with partition_cols creates a folder structure like: /year=1989/data.parquet
df_tourism.to_parquet(
    parquet_path,
    index=False,
    partition_cols=['year'],
    compression='snappy'
)
print(f"   -> Successfully partitioned data at: {parquet_path}")



# 2. SQL EXPORT (Data Warehouse Simulation)
print("\n2. Loading into SQLite...")

# Connect to SQLite (creates file if not exists)
conn = sqlite3.connect(db_path)

# Write to table 'fact_tourism_arrivals'
# if_exists='replace' refreshes the table on every run
try:
    df_tourism.to_sql('fact_tourism_arrivals', conn, if_exists='replace', index=False)
    
    # Validation Query
    cursor = conn.cursor()
    cursor.execute("SELECT year, sum(arrivals) FROM fact_tourism_arrivals WHERE year >= 2020 GROUP BY year")
    print("   -> Validation Query (Totals 2020+):")
    for row in cursor.fetchall():
        print(f"      {row}")
        
    print(f"   -> Database stored at: {db_path}")
finally:
    conn.close()


# 3. CSV EXPORT
print("\n3. Exporting CSV files...")

# Compressed (for sharing)
csv_path = f"{processed_path}/tourism_consolidated.csv.gz"
df_tourism.to_csv(csv_path, index=False, compression='gzip', sep=';', encoding='utf-8')
print(f"   -> Compressed CSV saved at: {csv_path}")

# Plain (Power BI compatible)
csv_plain_path = f"{processed_path}/tourism_consolidated.csv"
df_tourism.to_csv(csv_plain_path, index=False, sep=';', encoding='utf-8')
print(f"   -> Plain CSV saved at: {csv_plain_path}")

print("\nETL Pipeline finished successfully!")

1. Exporting to Parquet ...
   -> Successfully partitioned data at: ../data/processed/parquet_partitioned

2. Loading into SQLite...
   -> Validation Query (Totals 2020+):
      (2020, 2146435)
      (2021, 745871)
      (2022, 3630031)
      (2023, 5908341)
      (2024, 6773619)
   -> Database stored at: ../data/processed/tourism_dw.db

3. Exporting CSV files...
   -> Compressed CSV saved at: ../data/processed/tourism_consolidated.csv.gz
   -> Plain CSV saved at: ../data/processed/tourism_consolidated.csv

ETL Pipeline finished successfully!


## 6. Post-Export Sanity Check <a id="final-check"></a>
Final validation to ensure the exported layers are consistent and readable.

In [19]:
# --- Parquet Validation ---
print("Verifying Parquet (Gold Layer)...")
df_check = pd.read_parquet(parquet_path)
assert not df_check.isnull().values.any(), "Sanity Check Failed: NaN values found!"
assert df_check['arrivals'].dtype in ['int64', 'float64'], "Sanity Check Failed: Wrong dtype!"
print(f"   -> OK: {df_check.shape[0]} rows, {df_check.shape[1]} columns.")
print(f"   -> Schema: {list(df_check.columns)}")

# --- CSV Validation ---
print("\nVerifying Plain CSV (BI Layer)...")
df_csv_check = pd.read_csv(csv_plain_path, sep=';', nrows=5)
print(f"   -> OK: {df_csv_check.shape[1]} columns detected.")
print(f"   -> Preview:") 
display(df_csv_check.head(3))

print("\nAll exports validated successfully!")

Verifying Parquet (Gold Layer)...
   -> OK: 953672 rows, 9 columns.
   -> Schema: ['continent', 'country', 'state', 'arrival_method', 'month', 'arrivals', 'source_file', 'data_quality_flag', 'year']

Verifying Plain CSV (BI Layer)...
   -> OK: 9 columns detected.
   -> Preview:


,continent,country,state,arrival_method,year,month,arrivals,source_file,data_quality_flag
0,África,África Do Sul,Amazonas,Aéreo,1989,Janeiro,9,chegadas_1989.csv,original
1,África,Angola,Amazonas,Aéreo,1989,Janeiro,0,chegadas_1989.csv,original
2,África,Nigéria,Amazonas,Aéreo,1989,Janeiro,0,chegadas_1989.csv,original



All exports validated successfully!
